# JAX FFT comparisons and original CPU validation

This notebook compares the same input bytes under ordinary PyCBC CPU execution, default JAX execution, and a JAX scheme selecting only `fft` or `ifft` for original CPU execution. The default stays on the selected JAX device. Native validation asserts exact output dtype, shape and bytes against the selected CPU backend.

The examples cover real and complex single/double precision, seeded random values, cancellation, forward/inverse directions, plans and series normalization. Recorded results apply to the versions and device below; they establish operation-level behavior, not complete-search equivalence. Install PyCBC, JAX and Jupyter in the same environment.

For a **self-contained NumPy/JAX demonstration** without PyCBC or LAL, run
[fft rounding](jax_fft_rounding.ipynb). That notebook illustrates arithmetic
with generated inputs. This notebook checks the actual PyCBC implementations
and original validation routes. Results apply to each notebook's recorded
libraries and device.


In [1]:
import platform
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as np
import scipy
import jax
import jaxlib
import pycbc
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array, TimeSeries, FrequencySeries, zeros, aligned
from pycbc.fft import fft, ifft, FFT, IFFT, backend_cpu

DEVICE = "cpu"  # Use "cuda:0" to repeat on an available NVIDIA GPU.
CPU_BACKEND = backend_cpu.get_backend().__name__
with JAXScheme(DEVICE) as context:
    metadata = {
        "Python": platform.python_version(), "CPU architecture": platform.machine(),
        "PyCBC": pycbc.__version__, "NumPy": np.__version__,
        "SciPy": scipy.__version__, "JAX": jax.__version__,
        "jaxlib": jaxlib.__version__, "CPU FFT backend": CPU_BACKEND,
        "JAX platform": context.jax_device.platform,
        "JAX device kind": context.jax_device.device_kind,
        "JAX x64": jax.config.jax_enable_x64,
    }
assert metadata["JAX x64"], "Double-precision examples require JAX x64."
for name, value in metadata.items():
    print(f"{name}: {value}")


Python: 3.13.3
CPU architecture: x86_64
PyCBC: 2.12.dev1
NumPy: 2.2.5
SciPy: 1.15.2
JAX: 0.11.2
jaxlib: 0.11.2
CPU FFT backend: pycbc.fft.fftw
JAX platform: cpu
JAX device kind: cpu
JAX x64: True


## Fixed inputs and exact native assertions

Each call constructs fresh PyCBC buffers. CPU transform buffers use PyCBC aligned storage so their FFTW alignment matches the native adapter scratch buffers. The inverse input is the saved **CPU forward output**, so a forward discrepancy cannot be mistaken for an inverse discrepancy. All comparisons use the same output dtype and logical transform length.

Absolute error is reported alongside maximum error divided by the largest CPU output magnitude. This scale measure avoids dividing each near-zero frequency bin by its own value. Exact mismatch counts include distinct bit patterns such as signed zeros.

In [2]:
def assert_exact(actual, expected):
    assert actual.shape == expected.shape
    assert actual.dtype == expected.dtype
    assert actual.tobytes() == expected.tobytes()


def transform(direction, values, output_dtype, n, *, cpu=False, reference=(),
              plan=False, nbatch=1):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference)
    output_size = n // 2 + 1 if (direction == "fft"
                                and values.dtype.kind == "f") else n
    with context:
        source = zeros(values.size, dtype=values.dtype)
        source[:] = Array(values)
        destination = zeros(nbatch * output_size, dtype=output_dtype)
        if plan:
            cls = FFT if direction == "fft" else IFFT
            cls(source, destination, nbatch=nbatch,
                size=n if nbatch > 1 else None).execute()
        else:
            assert nbatch == 1
            (fft if direction == "fft" else ifft)(source, destination)
        return np.array(destination.numpy(), copy=True)


checks = []
def compare(name, direction, values, output_dtype, n, **kwargs):
    cpu = transform(direction, values, output_dtype, n, cpu=True, **kwargs)
    default = transform(direction, values, output_dtype, n, **kwargs)
    native = transform(direction, values, output_dtype, n,
                       reference=(direction,), **kwargs)
    assert_exact(native, cpu)
    error = np.max(np.abs(default - cpu))
    scale = max(float(np.max(np.abs(cpu))), np.finfo(cpu.real.dtype).tiny)
    item_bytes = cpu.dtype.itemsize
    mismatches = np.count_nonzero(np.any(
        cpu.view(np.uint8).reshape(-1, item_bytes)
        != default.view(np.uint8).reshape(-1, item_bytes), axis=1))
    checks.append(name)
    print(f"{name}: input={values.dtype}, output={cpu.dtype}, N={n}; "
          f"exact mismatches={mismatches}/{cpu.size}; "
          f"max abs={error:.6g}; scaled max={error / scale:.6g}; native exact")
    return cpu, default, native


rng = np.random.default_rng(1729)
inputs = {}
for dtype in (np.float32, np.float64, np.complex64, np.complex128):
    for n in (30, 32):
        random_values = rng.standard_normal(n)
        if np.dtype(dtype).kind == "c":
            random_values = random_values + 1j * rng.standard_normal(n)
        inputs[(np.dtype(dtype).name, n, "random")] = random_values.astype(dtype)
        large = 1e8 if dtype in (np.float32, np.complex64) else 1e16
        pattern = np.resize(np.array([large, 1, -large, 1]), n)
        if np.dtype(dtype).kind == "c":
            pattern = pattern + 1j * pattern[::-1]
        inputs[(np.dtype(dtype).name, n, "cancellation")] = pattern.astype(dtype)


## Forward and inverse comparisons

Both paths evaluate the discrete Fourier transform. Different FFT decompositions, operation orders or approximations to complex phase factors can produce different rounding. Cancellation makes those differences visible, especially in small output bins. This comparison does not reveal the exact internal butterfly or phase-factor calculation that differs in an individual bin.

JAX x64 enables double-precision arrays; it does not automatically widen a transform of single-precision inputs. Library choice, versions, planning and device execution remain relevant.

In [3]:
saved_forward = {}
for (dtype_name, n, pattern), values in inputs.items():
    spectrum_dtype = np.complex64 if dtype_name in ("float32", "complex64") else np.complex128
    label = f"{dtype_name}/{pattern}/N={n}"
    cpu, default, native = compare(label + "/fft", "fft", values,
                                  spectrum_dtype, n)
    saved_forward[(dtype_name, n, pattern)] = (cpu, default, native)
    _ = compare(label + "/ifft fixed input", "ifft", cpu, values.dtype, n)
assert len(checks) == 32


float32/random/N=30/fft: input=float32, output=complex64, N=30; exact mismatches=12/16; max abs=9.55535e-07; scaled max=9.88261e-08; native exact
float32/random/N=30/ifft fixed input: input=complex64, output=float32, N=30; exact mismatches=23/30; max abs=7.62939e-06; scaled max=1.16677e-07; native exact
float32/cancellation/N=30/fft: input=float32, output=complex64, N=30; exact mismatches=12/16; max abs=71.5542; scaled max=7.47945e-08; native exact
float32/cancellation/N=30/ifft fixed input: input=complex64, output=float32, N=30; exact mismatches=11/30; max abs=512; scaled max=1.70667e-07; native exact


float32/random/N=32/fft: input=float32, output=complex64, N=32; exact mismatches=5/17; max abs=4.76837e-07; scaled max=4.57007e-08; native exact
float32/random/N=32/ifft fixed input: input=complex64, output=float32, N=32; exact mismatches=20/32; max abs=3.8147e-06; scaled max=6.06766e-08; native exact
float32/cancellation/N=32/fft: input=float32, output=complex64, N=32; exact mismatches=8/17; max abs=0; scaled max=0; native exact
float32/cancellation/N=32/ifft fixed input: input=complex64, output=float32, N=32; exact mismatches=0/32; max abs=0; scaled max=0; native exact
float64/random/N=30/fft: input=float64, output=complex128, N=30; exact mismatches=12/16; max abs=1.25607e-15; scaled max=1.71681e-16; native exact
float64/random/N=30/ifft fixed input: input=complex128, output=float64, N=30; exact mismatches=22/30; max abs=1.06581e-14; scaled max=1.96043e-16; native exact
float64/cancellation/N=30/fft: input=float64, output=complex128, N=30; exact mismatches=12/16; max abs=11.6619; sca

float64/random/N=32/fft: input=float64, output=complex128, N=32; exact mismatches=9/17; max abs=8.88178e-16; scaled max=1.01721e-16; native exact
float64/random/N=32/ifft fixed input: input=complex128, output=float64, N=32; exact mismatches=21/32; max abs=7.10543e-15; scaled max=1.1876e-16; native exact
float64/cancellation/N=32/fft: input=float64, output=complex128, N=32; exact mismatches=8/17; max abs=0; scaled max=0; native exact
float64/cancellation/N=32/ifft fixed input: input=complex128, output=float64, N=32; exact mismatches=0/32; max abs=0; scaled max=0; native exact
complex64/random/N=30/fft: input=complex64, output=complex64, N=30; exact mismatches=26/30; max abs=1.2517e-06; scaled max=9.0146e-08; native exact


complex64/random/N=30/ifft fixed input: input=complex64, output=complex64, N=30; exact mismatches=28/30; max abs=8.52992e-06; scaled max=9.47817e-08; native exact
complex64/cancellation/N=30/fft: input=complex64, output=complex64, N=30; exact mismatches=24/30; max abs=68.352; scaled max=3.57727e-08; native exact
complex64/cancellation/N=30/ifft fixed input: input=complex64, output=complex64, N=30; exact mismatches=30/30; max abs=573.395; scaled max=1.91132e-07; native exact
complex64/random/N=32/fft: input=complex64, output=complex64, N=32; exact mismatches=22/32; max abs=9.61096e-07; scaled max=6.43112e-08; native exact
complex64/random/N=32/ifft fixed input: input=complex64, output=complex64, N=32; exact mismatches=26/32; max abs=6.1065e-06; scaled max=6.62513e-08; native exact
complex64/cancellation/N=32/fft: input=complex64, output=complex64, N=32; exact mismatches=0/32; max abs=0; scaled max=0; native exact
complex64/cancellation/N=32/ifft fixed input: input=complex64, output=comp

complex128/random/N=32/ifft fixed input: input=complex128, output=complex128, N=32; exact mismatches=26/32; max abs=1.06581e-14; scaled max=1.13206e-16; native exact
complex128/cancellation/N=32/fft: input=complex128, output=complex128, N=32; exact mismatches=0/32; max abs=0; scaled max=0; native exact
complex128/cancellation/N=32/ifft fixed input: input=complex128, output=complex128, N=32; exact mismatches=0/32; max abs=0; scaled max=0; native exact


## Precision control with identical rounded input

Widen a stored float32 input without regenerating it. The following reports how the single-precision CPU and JAX results differ from a double-precision JAX transform of those same values. The wider transform is a diagnostic, not a substitute for the original CPU target.

In [4]:
values = inputs[("float32", 30, "random")]
cpu_single, jax_single, _ = saved_forward[("float32", 30, "random")]
wide = transform("fft", values.astype(np.float64), np.complex128, 30)
for name, result in (("CPU single", cpu_single), ("JAX single", jax_single)):
    print(name, "maximum difference from widened-input JAX:",
          np.max(np.abs(result.astype(np.complex128) - wide)))


CPU single maximum difference from widened-input JAX: 1.118145654702324e-06
JAX single maximum difference from widened-input JAX: 1.157810081759496e-06


## Independently selecting directions and the full transform pair

Selecting only `ifft` reproduces the CPU inverse **of the supplied spectrum**, which may already contain a JAX forward discrepancy. Selecting both `fft` and `ifft` restores the original transform pair. A pair of plain Array transforms is unnormalized and should reproduce `N * input` within transform error, not the input itself.

In [5]:
values = inputs[("float64", 30, "random")]
cpu_f, jax_f, native_f = saved_forward[("float64", 30, "random")]
def complete_pair(values, *, cpu=False, reference=()):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference)
    with context:
        source = zeros(values.size, dtype=values.dtype)
        source[:] = Array(values)
        spectrum = zeros(values.size // 2 + 1, dtype=np.complex128)
        restored = zeros(values.size, dtype=np.float64)
        fft(source, spectrum)
        ifft(spectrum, restored)
        return np.array(restored.numpy(), copy=True)

cpu_pair = complete_pair(values, cpu=True)
native_pair = complete_pair(values, reference=("fft", "ifft"))
assert_exact(native_pair, cpu_pair)
checks.append("both-native transform pair")
# This checks the inverse boundary even when its input came from JAX.
cpu_inverse_of_jax = transform("ifft", jax_f, np.float64, 30, cpu=True)
native_inverse_of_jax = transform("ifft", jax_f, np.float64, 30,
                                  reference=("ifft",))
assert_exact(native_inverse_of_jax, cpu_inverse_of_jax)
checks.append("native inverse of JAX spectrum")
np.testing.assert_allclose(cpu_pair, 30 * values, rtol=1e-12, atol=1e-12)
print("Both-native pair matches CPU exactly; plain Array roundtrip scale is N.")
print("Native inverse of a JAX spectrum matches CPU on that same spectrum exactly.")


Both-native pair matches CPU exactly; plain Array roundtrip scale is N.
Native inverse of a JAX spectrum matches CPU on that same spectrum exactly.


## Batched class plans

Plans bind flattened consecutive rows to input/output buffers. Here two transforms each have logical size 32. The CPU plan and native plan use identical batch geometry. The selected CPU backend must support it; the original NumPy plan backend does not provide this batched comparison.

Plan execution supplies raw unnormalized transforms; series normalization belongs to the functional interface.

In [6]:
if CPU_BACKEND.endswith(".npfft"):
    print("Batched native-plan comparison unavailable for the selected NumPy CPU backend.")
else:
    for dtype in (np.float32, np.float64, np.complex64, np.complex128):
        values = rng.standard_normal(64)
        if np.dtype(dtype).kind == "c":
            values = values + 1j * rng.standard_normal(64)
        values = values.astype(dtype)
        spectrum_dtype = np.complex64 if dtype in (np.float32, np.complex64) else np.complex128
        cpu, _, _ = compare(f"{np.dtype(dtype).name}/batched plan fft", "fft",
                            values, spectrum_dtype, 32, plan=True, nbatch=2)
        _ = compare(f"{np.dtype(dtype).name}/batched plan ifft", "ifft",
                    cpu, values.dtype, 32, plan=True, nbatch=2)


float32/batched plan fft: input=float32, output=complex64, N=32; exact mismatches=23/34; max abs=6.7435e-07; scaled max=6.53212e-08; native exact
float32/batched plan ifft: input=complex64, output=float32, N=32; exact mismatches=20/64; max abs=4.29153e-06; scaled max=5.49358e-08; native exact
float64/batched plan fft: input=float64, output=complex128, N=32; exact mismatches=21/34; max abs=1.25607e-15; scaled max=1.03808e-16; native exact
float64/batched plan ifft: input=complex128, output=float64, N=32; exact mismatches=27/64; max abs=1.06581e-14; scaled max=1.64696e-16; native exact
complex64/batched plan fft: input=complex64, output=complex64, N=32; exact mismatches=50/64; max abs=2.03705e-06; scaled max=1.04409e-07; native exact
complex64/batched plan ifft: input=complex64, output=complex64, N=32; exact mismatches=45/64; max abs=7.8642e-06; scaled max=8.73277e-08; native exact


complex128/batched plan fft: input=complex128, output=complex128, N=32; exact mismatches=49/64; max abs=2.66454e-15; scaled max=1.66688e-16; native exact


complex128/batched plan ifft: input=complex128, output=complex128, N=32; exact mismatches=45/64; max abs=2.92964e-14; scaled max=4.00409e-16; native exact


## Series scale, spacing and epoch

The functional interface multiplies a TimeSeries forward transform by `delta_t`, and a FrequencySeries inverse transform by `delta_f`. With `delta_f = 1 / (N * delta_t)`, the unnormalized inverse factor N cancels. The examples below assert these scales, metadata and original CPU/native outputs separately.

The forward spectrum is saved **before** the inverse call: FFTW real inverse transforms may overwrite their CPU input buffer. The native JAX route uses a CPU scratch copy, preserving its immutable input. Comparing that overwritten buffer afterward would confuse input mutation with a forward-transform difference.

In [7]:
def series_pair(values, *, cpu=False, reference=()):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference)
    n = values.size
    dt = 1 / 2048
    cdtype = np.complex64 if values.dtype == np.float32 else np.complex128
    with context:
        source = TimeSeries(zeros(n, dtype=values.dtype), copy=False,
                            delta_t=dt, epoch=1000000000)
        source[:] = Array(values)
        spectrum = FrequencySeries(zeros(n // 2 + 1, dtype=cdtype),
                                   copy=False, delta_f=1)
        restored = TimeSeries(zeros(n, dtype=values.dtype), copy=False, delta_t=1)
        fft(source, spectrum)
        forward = spectrum.numpy().copy()
        ifft(spectrum, restored)
        assert spectrum.delta_f == 1 / (n * dt)
        assert restored.delta_t == dt
        assert spectrum.start_time == source.start_time == restored.start_time
        return (forward,
                np.array(restored.numpy(), copy=True))


for dtype in (np.float32, np.float64):
    values = np.arange(32, dtype=dtype)
    cpu_spectrum, cpu_restored = series_pair(values, cpu=True)
    jax_spectrum, jax_restored = series_pair(values)
    native_spectrum, native_restored = series_pair(values, reference=("fft", "ifft"))
    assert_exact(native_spectrum, cpu_spectrum)
    assert_exact(native_restored, cpu_restored)
    raw_cpu = transform("fft", values, cpu_spectrum.dtype, 32, cpu=True)
    np.testing.assert_array_equal(cpu_spectrum, raw_cpu * (1 / 2048))
    np.testing.assert_allclose(jax_restored, values, rtol=1e-6, atol=1e-6)
    checks.extend([f"{np.dtype(dtype).name}/series fft", f"{np.dtype(dtype).name}/series ifft"])
    print(dtype.__name__, "native spectrum/restored bytes match CPU; spacing/epoch passed;",
          "default roundtrip max error:", np.max(np.abs(jax_restored - values)))
print(f"Completed {len(checks)} exact CPU/native comparisons on the displayed device.")


float32 native spectrum/restored bytes match CPU; spacing/epoch passed; default roundtrip max error: 1.9073486e-06


float64 native spectrum/restored bytes match CPU; spacing/epoch passed; default roundtrip max error: 4.440892098500626e-15
Completed 46 exact CPU/native comparisons on the displayed device.


## Original CPU alignment as a separate control

FFTW can select a different algorithm when buffers are not aligned for SIMD. PyCBC passes its unaligned planning flag when either transform buffer misses the required alignment. Identical input values can then round differently even between two original CPU calls. This is independent of JAX arithmetic.

The next cell changes only the CPU input buffer alignment. It reports both layouts under the installed FFTW build, if available, and under the original NumPy provider. A zero difference is a valid measured result; builds need not choose different arithmetic. NumPy must match exactly for these identical inputs. For reproducible whole-search byte comparisons, select `--fft-backends numpy` in **both** searches. Validation switches honor that provider selection.

See the FFTW [alignment requirements](https://www.fftw.org/fftw3_doc/SIMD-alignment-and-fftw_005fmalloc.html) and [planner flags](https://www.fftw.org/fftw3_doc/Planner-Flags.html).

In [8]:
selected_backend = backend_cpu.cpu_backend
try:
    for provider in ("fftw", "numpy"):
        backend_cpu.set_backend([provider])
        if backend_cpu.cpu_backend != provider:
            print(provider, "unavailable in this installation")
            continue
        for dtype in (np.float32, np.float64):
            for n in (32, 512):
                values = np.random.default_rng(123).normal(size=n).astype(dtype)
                results = []
                with CPUScheme():
                    for offset in (0, 1):
                        storage = aligned.zeros(n + 1, dtype)
                        source = Array(storage[offset:offset + n], copy=False)
                        source[:] = Array(values)
                        assert source.ptr % pycbc.PYCBC_ALIGNMENT == (
                            0 if offset == 0 else np.dtype(dtype).itemsize)
                        output = zeros(n // 2 + 1, dtype=(
                            np.complex64 if dtype is np.float32 else np.complex128))
                        fft(source, output)
                        results.append(output.numpy().copy())
                if provider == "numpy":
                    assert_exact(results[0], results[1])
                print(provider, np.dtype(dtype).name, "N=", n,
                      "different bins=", np.count_nonzero(results[0] != results[1]),
                      "max abs=", np.max(np.abs(results[0] - results[1])))
finally:
    backend_cpu.set_backend([selected_backend])

fftw float32 N= 32 different bins= 12 max abs= 1.0662403e-06
fftw float32 N= 512 different bins= 237 max abs= 7.8642e-06
fftw float64 N= 32 different bins= 7 max abs= 9.930136612989092e-16
fftw float64 N= 512 different bins= 205 max abs= 1.4210854715202004e-14
numpy float32 N= 32 different bins= 0 max abs= 0.0
numpy float32 N= 512 different bins= 0 max abs= 0.0
numpy float64 N= 32 different bins= 0 max abs= 0.0
numpy float64 N= 512 different bins= 0 max abs= 0.0


## Limits and use

Select `reference_operations=("fft",)` or `("ifft",)` independently inside `JAXScheme`, or both names together. The CLI accepts `--jax-reference-operations fft,ifft`. These selections affect PyCBC's transform interfaces; direct raw JAX helper calls still use JAX.

Native routes transfer buffers to the host and run the selected original CPU backend. They can be much slower and prevent JAX tracing or differentiation through that boundary. Exact assertions here compare the same installed CPU backend and input bytes; other builds, devices and complete searches need their own comparisons.

See [JAX Fourier transforms](../../docs/jax_fft.rst) and [FFT numerical differences](../../docs/jax_fft_numerical_differences.rst).

## An inverse transform with its input held fixed

This comparison supplies identical complex64 frequency bytes to the original NumPy-backed PyCBC plan, default JAX plan and original `ifft` control. Preceding conditioning, PSD division and correlation cannot contribute. The printed NumPy return dtype distinguishes API-level precision promotion from a backend difference. Both stored outputs remain complex64. A differing result identifies this transform boundary; it does not identify an undocumented per-bin FFT instruction or imply that every device will differ.


In [9]:
import jax
import platform
try:
    WITNESS_DEVICE = "cuda:0" if jax.devices("gpu") else DEVICE
except RuntimeError:
    WITNESS_DEVICE = DEVICE
with JAXScheme(WITNESS_DEVICE) as witness_context:
    print({"JAX": jax.__version__, "NumPy": np.__version__, "CPU architecture": platform.machine(), "witness platform": witness_context.jax_device.platform, "witness device": witness_context.jax_device.device_kind})
# Freeze the complex64 frequency input so preceding correlation and PSD
# arithmetic cannot contribute to this transform comparison.
from pycbc.fft import backend_cpu, IFFT
from pycbc.types import Array, zeros

held_rng = np.random.default_rng(19073)
held_spectrum = (held_rng.standard_normal(256)
                 + 1j * held_rng.standard_normal(256)).astype(np.complex64)
held_backend = backend_cpu.cpu_backend

def held_plan_result(reference=None):
    context = CPUScheme() if reference is None else JAXScheme(WITNESS_DEVICE, reference_operations=reference)
    with context:
        source = Array(held_spectrum)
        target = zeros(len(source), dtype=np.complex64)
        IFFT(source, target).execute()
        return np.array(target.numpy(), copy=True)

try:
    backend_cpu.set_backend(['numpy'])
    held_cpu = held_plan_result()
    held_default = held_plan_result(())
    held_original = held_plan_result(('ifft',))
    assert held_original.dtype == held_default.dtype == held_cpu.dtype == np.complex64
    assert held_original.tobytes() == held_cpu.tobytes()
    print('NumPy FFT API return dtype:', np.fft.ifft(held_spectrum).dtype)
    print('Stored PyCBC inverse dtype:', held_cpu.dtype)
    print('Held-input differing complex values:', np.count_nonzero(held_cpu != held_default))
    print('Held-input maximum absolute difference:', np.max(np.abs(held_cpu.astype(np.complex128)
                                                                   - held_default.astype(np.complex128))))
finally:
    backend_cpu.set_backend([held_backend])


{'JAX': '0.11.2', 'NumPy': '2.2.5', 'CPU architecture': 'x86_64', 'witness platform': 'gpu', 'witness device': 'NVIDIA GeForce RTX 3090'}
NumPy FFT API return dtype: complex64
Stored PyCBC inverse dtype: complex64
Held-input differing complex values: 237
Held-input maximum absolute difference: 8.436110502698007e-06


You are using the class-based PyCBC FFT API, with the numpy backed. This is provided for convenience only. If performance is important use the class-based API with one of the other backends (for e.g. MKL or FFTW)
You are using the class-based PyCBC FFT API, with the numpy backed. This is provided for convenience only. If performance is important use the class-based API with one of the other backends (for e.g. MKL or FFTW)
